In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 55.6 MB/s eta 0:00:00


In [ ]:

!pip install -q condacolab
import condacolab
condacolab.install()


⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [ ]:

!conda install -c conda-forge tbb


Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - tbb


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2025.4.26  |       hbd8a1cb_0         149 KB  conda-forge
    certifi-2025.4.26          |     pyhd8ed1ab_0         154 KB  conda-forge
    conda-24.11.3              |  py311h38be061_0         1.1 MB  conda-forge
    libhwloc-2.11.2            |default_h0d58e46_1001         2.3 MB  conda-forge
    openssl-3.5.0              |       h7b32b05_1         3.0 MB  conda-forge
    tbb-2022.1.0               |       h4ce085d_0         175 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         6.9 MB

The following NEW packages will be INSTALLED:

  libhwloc           conda-for

In [ ]:
import numpy as np
from numba import njit, prange
import time

# ------------------------------------------------------------------------------
# Numba helper: extract candidate indices for one projection.
# ------------------------------------------------------------------------------
@njit(parallel=True)
def query_candidates_nb(q, sorted_projs, sorted_idxs, proj_dirs, window_size, n, L):
    """
    For a given query q (1D array, float32) and precomputed arrays,
    extract a window of candidate indices from each projection direction.

    Parameters:
      q : (d,) query point (float32)
      sorted_projs : (L, n) sorted projection values per direction.
      sorted_idxs  : (L, n) corresponding sorted indices.
      proj_dirs    : (L, d) projection directions.
      window_size  : half-window size (integer)
      n            : total number of points (integer)
      L            : number of projection directions (integer)

    Returns:
      candidate_indices : 1D array of unique candidate indices.
    """
    # Pre-allocate candidate matrix (L x (2*window_size))
    candidate_matrix = np.empty((L, 2 * window_size), dtype=np.int32)

    # Loop over each projection direction in parallel.
    for i in prange(L):
        # Compute dot product of q and the i-th projection direction.
        dot = 0.0
        for j in range(q.shape[0]):
            dot += proj_dirs[i, j] * q[j]
        q_val = dot
        # Binary search in the sorted projection array.
        idx = np.searchsorted(sorted_projs[i], q_val)
        slice_size = 2 * window_size
        start = idx - window_size
        if start < 0:
            start = 0
        if start > n - slice_size:
            start = n - slice_size
        # Extract candidate indices for this projection.
        candidate_matrix[i] = sorted_idxs[i, start:start + slice_size]

    # Flatten candidate_matrix and compute unique indices.
    flat = candidate_matrix.reshape(L * 2 * window_size)
    flat_sorted = np.sort(flat)
    # Count unique values.
    count = 1
    for i in range(1, flat_sorted.shape[0]):
        if flat_sorted[i] != flat_sorted[i - 1]:
            count += 1
    unique_arr = np.empty(count, dtype=flat_sorted.dtype)
    unique_arr[0] = flat_sorted[0]
    pos = 1
    for i in range(1, flat_sorted.shape[0]):
        if flat_sorted[i] != flat_sorted[i - 1]:
            unique_arr[pos] = flat_sorted[i]
            pos += 1
    return unique_arr

# ------------------------------------------------------------------------------
# Numba helper: compute squared Euclidean distances for candidate indices.
# ------------------------------------------------------------------------------
@njit
def compute_dists(q, data, candidate_indices):
    M = candidate_indices.shape[0]
    d = q.shape[0]
    dists = np.empty(M, dtype=np.float32)
    for i in range(M):
        s = 0.0
        for j in range(d):
            diff = data[candidate_indices[i], j] - q[j]
            s += diff * diff
        dists[i] = s
    return dists

# ------------------------------------------------------------------------------
# AMPIIndex: Adaptive Multi-Projection Index using NumPy and Numba.
# ------------------------------------------------------------------------------
class AMPIIndex:
    def __init__(self, data, num_projections):
        """
        Initializes the AMPI index.

        Args:
          data: A NumPy array of shape (n, d) containing n data points.
          num_projections: The number L of random projection directions.
        """
        self.data = data  # shape (n, d)
        self.n, self.d = data.shape
        self.L = num_projections

        rng = np.random.default_rng(0)
        # Generate L random projection directions (each normalized)
        raw_dirs = rng.standard_normal((self.L, self.d)).astype(np.float32)
        norms = np.linalg.norm(raw_dirs, axis=1, keepdims=True)
        self.proj_dirs = raw_dirs / norms  # shape (L, d)

        # Precompute projections for each direction.
        self.sorted_projs = np.empty((self.L, self.n), dtype=np.float32)
        self.sorted_idxs = np.empty((self.L, self.n), dtype=np.int32)
        for i in range(self.L):
            proj = data @ self.proj_dirs[i]  # (n,)
            sorted_idx = np.argsort(proj)
            self.sorted_projs[i] = proj[sorted_idx]
            self.sorted_idxs[i] = sorted_idx

    def query_candidates(self, q, window_size=10):
        """
        For query q, extract candidate indices using the projection-based search.

        Args:
          q: Query point of shape (d,) (NumPy array, float32)
          window_size: half-window size.

        Returns:
          candidate_indices: 1D NumPy array of unique candidate indices.
        """
        # Ensure q is a float32 1D array.
        q = np.asarray(q, dtype=np.float32)
        candidate_indices = query_candidates_nb(q, self.sorted_projs, self.sorted_idxs,
                                                self.proj_dirs, window_size, self.n, self.L)
        return candidate_indices

    def query(self, q, window_size=10, k=1):
        """
        Computes the k-NN result:
          1. Extract candidate indices.
          2. Compute full L2 distances on candidate set.
          3. Return top k neighbors.

        Returns:
          final_points: (k, d) array of k nearest neighbors.
          final_dists: (k,) array of squared L2 distances.
          candidate_indices: candidate set used.
        """
        candidate_indices = self.query_candidates(q, window_size=window_size)
        dists = compute_dists(q, self.data, candidate_indices)
        sorted_order = np.argsort(dists)
        final_indices = candidate_indices[sorted_order][:k]
        final_points = self.data[final_indices]
        final_dists = dists[sorted_order][:k]
        return final_points, final_dists, final_indices

In [ ]:
import numpy as np
import time
import faiss
from torchvision import datasets

def test_mnist(k=5, num_projections=128, window_size=128, num_queries=1_000):

    # Upload the dataset
    train_dataset = datasets.MNIST(root='./data', train=True, download=True)
    test_dataset  = datasets.MNIST(root='./data', train=False, download=True)

    # Flatten images and normalize pixel values.
    train_data = train_dataset.data.float().view(-1, 28*28) / 255.0
    test_data  = test_dataset.data.float().view(-1, 28*28) / 255.0
    X = np.concatenate([train_data, test_data], axis=0)  # shape: (70000, 784)
    n, d = X.shape

    # Build the FAISS index for exact L2 search.
    index_faiss = faiss.IndexFlatL2(d)
    index_faiss.add(X)

    # Build the AMPI index.
    ampi = AMPIIndex(X, num_projections)

    # Choose random queries.
    rng = np.random.default_rng(42)
    query_indices = rng.choice(n, size=num_queries, replace=False)

    error_ratios = []
    faiss_runtime = []
    ampi_runtime = []
    overlap_ratios = []  # List to store index overlap ratios

    for count, qi in enumerate(query_indices, start=1):
        q = X[qi]

        # FAISS exact k-NN search.
        start_faiss = time.perf_counter_ns()
        D_faiss, I_faiss = index_faiss.search(q.reshape(1, -1), k)
        end_faiss = time.perf_counter_ns()
        faiss_runtime.append(end_faiss - start_faiss)
        d_faiss = D_faiss[0]
        i_faiss = I_faiss[0]

        # AMPI approximate search.
        start_ampi = time.perf_counter_ns()
        approx_points, approx_dists, approx_indices = ampi.query(q, window_size, k)
        end_ampi = time.perf_counter_ns()
        ampi_runtime.append(end_ampi - start_ampi)
        d_ampi = np.array(approx_dists)
        i_ampi = np.array(approx_indices)

        print(f"Query {count}/{num_queries} (Index: {qi}):")
        print("  FAISS first neighbors -> distances:", d_faiss[:k], ", index:", i_faiss[:k])
        if len(d_ampi) > 0 and len(approx_indices) > 0:
            print("  AMPI first neighbors  -> distances:", d_ampi[:k], ", index:", approx_indices[:k])
        else:
            print("  AMPI returned no neighbors.")

        # Compute error ratio for the first neighbor, avoiding division by zero.
        ratio = np.divide(d_ampi, d_faiss, out=np.ones_like(d_ampi), where=(d_faiss != 0))
        error_ratios.append(np.mean(ratio))

        # Compute index overlap between FAISS and AMPI results.
        faiss_set = set(i_faiss[:k])
        ampi_set = set(approx_indices[:k])
        overlap = len(faiss_set.intersection(ampi_set))
        overlap_ratio = overlap / k
        overlap_ratios.append(overlap_ratio)
        print(f"  Index Overlap: {overlap} out of {k} (ratio: {overlap_ratio:.2f})\n")

    avg_ratio = np.mean(error_ratios)
    avg_faiss_time = np.mean(faiss_runtime)
    avg_ampi_time = np.mean(ampi_runtime)
    avg_overlap_ratio = np.mean(overlap_ratios)

    print(f"\nAverage index overlap ratio: {avg_overlap_ratio:.2f}")
    print(f"Average distance ratio (AMPI / FAISS): {avg_ratio:.6f}")
    print(f"FAISS average runtime per query: {avg_faiss_time/num_queries:.3f} ns")
    print(f"AMPI average runtime per query: {avg_ampi_time/num_queries:.3f} ns")

    tolerance = 1.10  # Allow up to 10% increase in distance.
    if avg_ratio > tolerance:
        print(f"AMPI's average distance ratio {avg_ratio:.6f} exceeds the tolerance {tolerance}")
    else:
        print(f"AMPI's average distance ratio {avg_ratio:.6f} is within the tolerance {tolerance}")

if __name__ == "__main__":
    # Use smaller numbers for quick testing; adjust parameters as needed.
    test_mnist(k=50, num_projections=256, window_size=128, num_queries=1000)

100%|██████████| 9.91M/9.91M [00:00<00:00, 42.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.13MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.96MB/s]


Streaming output truncated to the last 5000 lines.
 36089 18364]
  Index Overlap: 43 out of 50 (ratio: 0.86)

Query 818/1000 (Index: 15226):
  FAISS first neighbors -> distances: [ 0.       20.580978 21.587452 22.754665 23.384823 23.868572 25.118385
 25.243876 25.528244 25.55591  26.00057  26.535225 26.566998 26.626839
 26.635342 27.108421 27.433126 27.96463  28.207844 28.42173  28.719076
 28.778088 28.896317 29.206215 29.344545 29.653412 30.105593 30.122786
 30.169567 30.19128  30.913296 31.448996 31.523079 31.883368 31.964123
 32.120804 32.29706  32.34862  32.3777   32.471973 32.62681  33.00403
 33.112926 33.176178 33.212444 33.229485 33.491566 33.80609  33.813797
 33.916817] , index: [15226 54201 15497 53566 39686 15547 62069 41015 25231 60573 64004 43330
 48682  2751 32652 56420 47356 18612  5980 66863 53488 32944 41007 53474
 38827 21250 43076  6892 50658 15176 51816 15292  5942 18044 64996 58881
  8094  6216 69275  4508 43070 18752 65516 44188 56927 40883 64334 41085
  2660 16134

In [ ]:
import numpy as np
import time
import faiss

def test_randn(n=100_000, d=512, k=5, num_projections=128, window_size=128, num_queries=1_000):
    # Seed for reproducibility.
    np.random.seed(0)
    # Generate a random Gaussian dataset.
    X = np.random.randn(n, d).astype(np.float32)

    # Build the FAISS index for exact L2 search.
    index_faiss = faiss.IndexFlatL2(d)
    index_faiss.add(X)

    # Build the AMPI index.
    ampi = AMPIIndex(X, num_projections)

    # Choose random queries.
    rng = np.random.default_rng(42)
    query_indices = rng.choice(n, size=num_queries, replace=False)

    error_ratios = []
    faiss_runtime = []
    ampi_runtime = []
    overlap_ratios = []  # List to store index overlap ratios

    for count, qi in enumerate(query_indices, start=1):
        q = X[qi]

        # FAISS exact k-NN search.
        start_faiss = time.perf_counter_ns()
        D_faiss, I_faiss = index_faiss.search(q.reshape(1, -1), k)
        end_faiss = time.perf_counter_ns()
        faiss_runtime.append(end_faiss - start_faiss)
        d_faiss = D_faiss[0]
        i_faiss = I_faiss[0]

        # AMPI approximate search.
        start_ampi = time.perf_counter_ns()
        approx_points, approx_dists, approx_indices = ampi.query(q, window_size, k)
        end_ampi = time.perf_counter_ns()
        ampi_runtime.append(end_ampi - start_ampi)
        d_ampi = np.array(approx_dists)
        i_ampi = np.array(approx_indices)

        print(f"Query {count}/{num_queries} (Index: {qi}):")
        print("  FAISS first neighbors -> distances:", d_faiss[:k], ", index:", i_faiss[:k])
        if len(d_ampi) > 0 and len(approx_indices) > 0:
            print("  AMPI first neighbors  -> distances:", d_ampi[:k], ", index:", approx_indices[:k])
        else:
            print("  AMPI returned no neighbors.")

        # Compute error ratio for the first neighbor, avoiding division by zero.
        ratio = np.divide(d_ampi, d_faiss, out=np.ones_like(d_ampi), where=(d_faiss != 0))
        error_ratios.append(np.mean(ratio))

        # Compute index overlap between FAISS and AMPI results.
        faiss_set = set(i_faiss[:k])
        ampi_set = set(approx_indices[:k])
        overlap = len(faiss_set.intersection(ampi_set))
        overlap_ratio = overlap / k
        overlap_ratios.append(overlap_ratio)
        print(f"  Index Overlap: {overlap} out of {k} (ratio: {overlap_ratio:.2f})\n")

    avg_ratio = np.mean(error_ratios)
    avg_faiss_time = np.mean(faiss_runtime)
    avg_ampi_time = np.mean(ampi_runtime)
    avg_overlap_ratio = np.mean(overlap_ratios)

    print(f"\nAverage index overlap ratio: {avg_overlap_ratio:.2f}")
    print(f"Average distance ratio (AMPI / FAISS): {avg_ratio:.6f}")
    print(f"FAISS average runtime per query: {avg_faiss_time/num_queries:.3f} ns")
    print(f"AMPI average runtime per query: {avg_ampi_time/num_queries:.3f} ns")

    tolerance = 1.10  # Allow up to 10% increase in distance.
    if avg_ratio > tolerance:
        print(f"AMPI's average distance ratio {avg_ratio:.6f} exceeds the tolerance {tolerance}")
    else:
        print(f"AMPI's average distance ratio {avg_ratio:.6f} is within the tolerance {tolerance}")

if __name__ == "__main__":
    # Use smaller numbers for quick testing; adjust parameters as needed.
    test_randn(n=1_000_000, d=100, k=16, num_projections=1024, window_size=512, num_queries=1_000_000)

Streaming output truncated to the last 5000 lines.
 661189 362181 276976 405587 367065 453692]
  AMPI first neighbors  -> distances: [  0.       102.88522  102.997665 103.09758  106.47971  107.74876
 108.506485 109.717    110.84072  113.23992  113.35112  113.41457
 113.44942  113.51694  113.564285 113.576904] , index: [793127 383652 519044 298349 678352 573062 374223  53791 759927 661189
 362181 276976 405587 367065 453692 980418]
  Index Overlap: 15 out of 16 (ratio: 0.94)

Query 9035/1000000 (Index: 676120):
  FAISS first neighbors -> distances: [  0.       101.60219  104.28445  105.440094 106.06532  106.38544
 106.86155  107.04991  107.69072  107.88661  108.6627   109.1514
 109.15302  110.34449  111.05522  111.44986 ] , index: [676120 906958 840250 861945 947053 433126 783975 515722 642441 145522
 455978 185205 670448  46330 928490 628109]
  AMPI first neighbors  -> distances: [  0.       104.28445  105.440094 106.38544  106.86154  107.04992
 107.88662  108.6627   109.151405 109.153